In [ ]:
from google.colab import drive

drive.mount("/content/drive")

%pip install -q ultralytics==8.4.218

In [ ]:
import os
import sys
import cv2
import json
import time
import torch
import random
import hashlib
import importlib
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
from ultralytics import YOLO
from tqdm.auto import tqdm
from IPython.display import display

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

assert PROJECT_ROOT.is_dir()

SRC_DIR = PROJECT_ROOT / "src"
CONFIG_DIR = PROJECT_ROOT / "configs"

PHASE9B = PROJECT_ROOT / "results/pipeline_refinement"

SESSION_ID = datetime.now().strftime(
    "%Y%m%d-%H%M%S-%f"
)

SESSION_DIR = PHASE9B / SESSION_ID

TABLES_DIR = SESSION_DIR / "tables"
FIGURES_DIR = SESSION_DIR / "figures"
REPORTS_DIR = SESSION_DIR / "reports"
UPLOAD_DIR = SESSION_DIR / "uploads"
FRAMES_DIR = SESSION_DIR / "video_frames"
OUTPUT_DIR = SESSION_DIR / "outputs"

for folder in [
    TABLES_DIR,
    FIGURES_DIR,
    REPORTS_DIR,
    UPLOAD_DIR,
    FRAMES_DIR,
    OUTPUT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

APP_CONFIG_PATH = CONFIG_DIR / "app_config.json"

assert APP_CONFIG_PATH.is_file()

APP_CONFIG = json.loads(
    APP_CONFIG_PATH.read_text()
)

def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024**2), b""
        ):
            sha.update(chunk)

    return sha.hexdigest()


ORIGINAL_CONFIG_HASH = sha256_file(
    APP_CONFIG_PATH
)

for task, cfg in APP_CONFIG["models"].items():

    path = PROJECT_ROOT / cfg["weights"]

    assert path.is_file(), path

    assert sha256_file(path) == (
        cfg["model_sha256"]
    ), f"Integrity failed: {task}"

    print(task.upper(), "MODEL VERIFIED")

ENGINE_PATH = SRC_DIR / "trafficguard_engine.py"

VIDEO_MODULE = SRC_DIR / "video_processor.py"

assert ENGINE_PATH.is_file()
assert VIDEO_MODULE.is_file()

ORIGINAL_SOURCE_HASHES = {
    str(p): sha256_file(p)
    for p in [ENGINE_PATH, VIDEO_MODULE]
}

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from trafficguard_engine import TrafficGuardEngine

engine = TrafficGuardEngine(
    project_root=PROJECT_ROOT,
    device=DEVICE
)

print("\nDevice:", DEVICE)
print("Session directory:", SESSION_DIR)

print("\nPHASE 9B ENVIRONMENT READY.")

In [ ]:
import zipfile
import tempfile
import yaml

EXPECTED_DATASETS = {
    "helmet": "helmet_v3",
    "seatbelt": "seatbelt_v1"
}

DATASET_ROOTS = {}

def locate_dataset(version):

    direct = (
        PROJECT_ROOT /
        "datasets/processed" /
        version
    )

    def valid(path):

        return (
            (path / "val/images").is_dir()
            and (path / "val/labels").is_dir()
        )

    if valid(direct):
        return direct

    candidates = [
        p.parent.parent
        for p in PROJECT_ROOT.rglob("val/images")
        if version in p.parts
        and valid(p.parent.parent)
    ]

    candidates = sorted(set(candidates))

    if len(candidates) == 1:
        return candidates[0]

    archives = list(
        PROJECT_ROOT.rglob(f"{version}.zip")
    )

    if not candidates and len(archives) == 1:

        destination = Path(
            tempfile.mkdtemp(
                prefix=f"{version}_"
            )
        )

        with zipfile.ZipFile(archives[0]) as z:

            for member in z.infolist():

                target = (
                    destination / member.filename
                ).resolve()

                assert target.is_relative_to(
                    destination.resolve()
                )

            z.extractall(destination)

        candidates = [
            p.parent.parent
            for p in destination.rglob("val/images")
            if valid(p.parent.parent)
        ]

    assert len(candidates) == 1, (
        f"Cannot uniquely locate {version}: "
        f"{candidates}"
    )

    return candidates[0]


def read_label_rows(path):

    rows = []

    for line in path.read_text().splitlines():

        if not line.strip():
            continue

        values = list(map(float, line.split()))

        assert len(values) == 5
        assert values[0] in [0, 1]

        rows.append(values)

    return rows


inventory = []

for task, version in EXPECTED_DATASETS.items():

    root = locate_dataset(version)

    DATASET_ROOTS[task] = root

    images = sorted([
        p for p in (root / "val/images").iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ])

    for image_path in images:

        label_path = (
            root / "val/labels" /
            f"{image_path.stem}.txt"
        )

        assert label_path.is_file()

        rows = read_label_rows(label_path)

        class0 = sum(
            int(row[0]) == 0 for row in rows
        )

        class1 = sum(
            int(row[0]) == 1 for row in rows
        )

        if class0 and class1:
            category = "mixed"

        elif class0:
            category = "compliant_only"

        elif class1:
            category = "violation_only"

        else:
            category = "background"

        inventory.append({
            "task": task,
            "dataset": version,
            "image_path": str(image_path),
            "label_path": str(label_path),
            "expected_class0": class0,
            "expected_class1": class1,
            "category": category
        })

    print(
        task, "validation images:", len(images)
    )

inventory_df = pd.DataFrame(inventory)

inventory_df.to_csv(
    TABLES_DIR / "validation_image_inventory.csv",
    index=False
)

display(
    inventory_df.groupby(
        ["task", "category"]
    ).size().rename("Images").reset_index()
)

assert len(
    inventory_df.query("task == 'helmet'")
) == 305

assert len(
    inventory_df.query("task == 'seatbelt'")
) == 787

print("\nDATASET INDEXING COMPLETED.")

In [ ]:
SAMPLES_PER_CATEGORY = 3

sample_groups = []

for (task, category), group in inventory_df.groupby(
    ["task", "category"]
):

    selected = group.sample(
        n=min(SAMPLES_PER_CATEGORY, len(group)),
        random_state=SEED
    )

    sample_groups.append(selected)

diagnostic_df = pd.concat(
    sample_groups,
    ignore_index=True
)

diagnostic_df.to_csv(
    TABLES_DIR / "diagnostic_sample_manifest.csv",
    index=False
)

display(
    diagnostic_df[
        [
            "task",
            "category",
            "expected_class0",
            "expected_class1"
        ]
    ]
)

print(
    "Selected diagnostic images:",
    len(diagnostic_df)
)

In [ ]:
def draw_ground_truth(image, label_path, task):

    output = image.copy()

    h, w = output.shape[:2]

    names = {
        0: f"with_{task}",
        1: f"without_{task}"
    }

    for cls, x, y, bw, bh in read_label_rows(
        Path(label_path)
    ):

        cls = int(cls)

        x1 = int((x-bw/2)*w)
        y1 = int((y-bh/2)*h)

        x2 = int((x+bw/2)*w)
        y2 = int((y+bh/2)*h)

        color = (
            (0, 180, 0)
            if cls == 0
            else (0, 0, 255)
        )

        cv2.rectangle(
            output,
            (x1, y1),
            (x2, y2),
            color,
            2
        )

        cv2.putText(
            output,
            names[cls],
            (x1, max(15, y1-5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            color,
            1
        )

    return output


for task in ["helmet", "seatbelt"]:

    subset = diagnostic_df[
        diagnostic_df["task"] == task
    ].reset_index(drop=True)

    rows = int(np.ceil(len(subset)/3))

    fig, axes = plt.subplots(
        rows, 3,
        figsize=(15, rows*4),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for i, row in subset.iterrows():

        image = cv2.imread(row["image_path"])

        assert image is not None

        annotated = draw_ground_truth(
            image,
            row["label_path"],
            task
        )

        ax = axes.flat[i]

        ax.imshow(
            cv2.cvtColor(
                annotated,
                cv2.COLOR_BGR2RGB
            )
        )

        ax.set_title(
            f'{row["category"]}\n'
            f'With: {row["expected_class0"]} | '
            f'Without: {row["expected_class1"]}'
        )

    plt.suptitle(
        f"{task.title()} — Diagnostic Ground Truth"
    )

    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR /
        f"{task}_diagnostic_ground_truth.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

In [ ]:
REFINED_CODE = r'''
import cv2
import torch
import time
import numpy as np

from trafficguard_engine import TrafficGuardEngine


PROFILES = {

    "standard": {
        "helmet_size": 640,
        "helmet_conf": 0.30,
        "seatbelt_size": 640,
        "seatbelt_conf": 0.35,
        "roi": False
    },

    "high_resolution": {
        "helmet_size": 800,
        "helmet_conf": 0.30,
        "seatbelt_size": 640,
        "seatbelt_conf": 0.35,
        "roi": False
    },

    "maximum_resolution": {
        "helmet_size": 960,
        "helmet_conf": 0.30,
        "seatbelt_size": 640,
        "seatbelt_conf": 0.35,
        "roi": False
    },

    "high_sensitivity": {
        "helmet_size": 800,
        "helmet_conf": 0.20,
        "seatbelt_size": 640,
        "seatbelt_conf": 0.35,
        "roi": False
    },

    "region_assisted": {
        "helmet_size": 640,
        "helmet_conf": 0.30,
        "seatbelt_size": 640,
        "seatbelt_conf": 0.35,
        "roi": True
    }
}


class RefinedTrafficGuard(TrafficGuardEngine):

    def __init__(
        self,
        project_root,
        device=None,
        profile="standard"
    ):

        super().__init__(
            project_root,
            device=device
        )

        self.set_profile(profile)

        self._current_context = []


    def set_profile(self, name):

        if name not in PROFILES:
            raise ValueError(
                f"Unknown profile: {name}"
            )

        with self.lock:

            self.profile_name = name
            self.profile = PROFILES[name].copy()


    @classmethod
    def iou(cls, a, b):

        intersection = cls.intersection(a, b)

        union = (
            cls.area(a)
            + cls.area(b)
            - intersection
        )

        return intersection / max(union, 1e-9)


    def suppress_duplicates(
        self,
        detections,
        threshold=0.5
    ):

        kept = []

        ordered = sorted(
            detections,
            key=lambda d: d["confidence"],
            reverse=True
        )

        for detection in ordered:

            duplicate = any(
                detection["class_id"] == old["class_id"]
                and self.iou(
                    detection["bbox"],
                    old["bbox"]
                ) >= threshold

                for old in kept
            )

            if not duplicate:
                kept.append(detection)

        return kept


    def raw_predict(
        self,
        task,
        image,
        size=None,
        confidence=None
    ):

        cfg = self.config["models"][task]

        kwargs = {
            "source": image,
            "imgsz": (
                size if size is not None
                else cfg["image_size"]
            ),
            "conf": (
                confidence if confidence is not None
                else cfg["confidence"]
            ),
            "iou": cfg["nms_iou"],
            "max_det": cfg.get("max_det", 300),
            "device": self.device,
            "batch": 1,
            "verbose": False,
            "save": False
        }

        if task == "context":

            kwargs["classes"] = cfg["classes"]

        with self.lock, torch.inference_mode():

            result = self.models[task].predict(
                **kwargs
            )[0]

            detections = []

            if result.boxes is not None:

                for values in (
                    result.boxes.data
                    .detach()
                    .cpu()
                    .numpy()
                ):

                    x1, y1, x2, y2, conf, cls = values

                    cls = int(cls)

                    detections.append({
                        "task": task,
                        "class_id": cls,
                        "label": self.models[task].names[cls],
                        "confidence": float(conf),
                        "bbox": [
                            float(x1),
                            float(y1),
                            float(x2),
                            float(y2)
                        ],
                        "source": "full_image"
                    })

        return detections


    def motorcycle_crops(
        self,
        image,
        context,
        full_detections
    ):

        height, width = image.shape[:2]

        motorcycles = [
            d for d in context
            if d["class_id"] == 3
        ]

        # Limit expensive additional inference.
        motorcycles = sorted(
            motorcycles,
            key=lambda d: d["confidence"],
            reverse=True
        )[:4]

        extra = []

        for motorcycle in motorcycles:

            x1, y1, x2, y2 = motorcycle["bbox"]

            bw = x2-x1
            bh = y2-y1

            # Expand upwards to include riders.
            cx1 = max(0, int(x1-0.40*bw))
            cy1 = max(0, int(y1-1.50*bh))

            cx2 = min(width, int(x2+0.40*bw))
            cy2 = min(height, int(y2+0.10*bh))

            region = [cx1, cy1, cx2, cy2]

            if cx2-cx1 < 32 or cy2-cy1 < 32:
                continue

            # Avoid extra inference when a head
            # detection is already present nearby.
            supported_head = any(
                self.coverage(
                    d["bbox"], region
                ) >= 0.8
                for d in full_detections
            )

            if supported_head:
                continue

            crop = image[cy1:cy2, cx1:cx2]

            rows = self.raw_predict(
                "helmet",
                crop,
                size=640,
                confidence=self.profile["helmet_conf"]
            )

            for row in rows:

                a, b, c, d = row["bbox"]

                row["bbox"] = [
                    a+cx1,
                    b+cy1,
                    c+cx1,
                    d+cy1
                ]

                row["source"] = "motorcycle_crop"

                extra.append(row)

        return extra


    def predict_model(self, task, image):

        if task == "context":

            self._current_context = (
                self.raw_predict(task, image)
            )

            return self._current_context

        rows = self.raw_predict(
            task,
            image,
            size=self.profile[f"{task}_size"],
            confidence=self.profile[f"{task}_conf"]
        )

        if task == "helmet" and self.profile["roi"]:

            extra = self.motorcycle_crops(
                image,
                self._current_context,
                rows
            )

            rows = self.suppress_duplicates(
                rows + extra
            )

        return rows


    def process_image(
        self,
        image,
        mode="combined",
        use_context=True
    ):

        with self.lock:

            self._current_context = []

            result = super().process_image(
                image,
                mode=mode,
                use_context=use_context
            )

            result["summary"]["profile"] = (
                self.profile_name
            )

            result["summary"]["crop_detections"] = sum(
                d.get("source") == "motorcycle_crop"
                for d in result["detections"]
            )

            self._current_context = []

            return result


    def draw_raw(self, image, detections):

        output = image.copy()

        for d in detections:

            x1, y1, x2, y2 = map(
                int, d["bbox"]
            )

            if d["task"] == "context":

                color = (255, 180, 0)

            else:

                color = (
                    (0, 180, 0)
                    if d["class_id"] == 0
                    else (0, 0, 255)
                )

            cv2.rectangle(
                output,
                (x1, y1),
                (x2, y2),
                color,
                2
            )

            label = (
                f'{d["label"]} '
                f'{d["confidence"]:.2f}'
            )

            cv2.putText(
                output,
                label,
                (x1, max(15, y1-5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                color,
                1
            )

        return output
'''

REFINED_PATH = SRC_DIR / "trafficguard_refined.py"

compile(
    REFINED_CODE,
    str(REFINED_PATH),
    "exec"
)

if REFINED_PATH.exists():

    assert REFINED_PATH.read_text() == REFINED_CODE, (
        "A different refinement file already exists. "
        "Preserve or version it before replacing."
    )

else:

    REFINED_PATH.write_text(
        REFINED_CODE,
        encoding="utf-8"
    )

print("Refined engine created:", REFINED_PATH)
print("Python syntax: PASSED")

In [ ]:
import gc

# Release the previous engine before loading
# another copy of the three models.

globals().pop("engine", None)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

import trafficguard_refined

importlib.reload(trafficguard_refined)

from trafficguard_refined import (
    RefinedTrafficGuard,
    PROFILES
)

engine = RefinedTrafficGuard(
    project_root=PROJECT_ROOT,
    device=DEVICE,
    profile="standard"
)

display(
    pd.DataFrame(PROFILES).T
)

print("\nREFINED ENGINE READY.")
print("Active profile:", engine.profile_name)
print("Device:", engine.device)

In [ ]:
def ground_truth_objects(image, label_path):

    h, w = image.shape[:2]

    ground = []

    for cls, x, y, bw, bh in read_label_rows(
        Path(label_path)
    ):

        ground.append({
            "class_id": int(cls),
            "bbox": [
                (x-bw/2)*w,
                (y-bh/2)*h,
                (x+bw/2)*w,
                (y+bh/2)*h
            ]
        })

    return ground


def match_detections(
    ground,
    predictions,
    class_id,
    iou_threshold=0.50
):

    targets = [
        d for d in ground
        if d["class_id"] == class_id
    ]

    preds = sorted(
        [
            d for d in predictions
            if d["class_id"] == class_id
        ],
        key=lambda d: d["confidence"],
        reverse=True
    )

    matched = set()

    tp = fp = 0

    for pred in preds:

        available = [
            i for i in range(len(targets))
            if i not in matched
        ]

        if not available:

            fp += 1
            continue

        best = max(
            available,
            key=lambda i: engine.iou(
                pred["bbox"],
                targets[i]["bbox"]
            )
        )

        overlap = engine.iou(
            pred["bbox"],
            targets[best]["bbox"]
        )

        if overlap >= iou_threshold:

            tp += 1
            matched.add(best)

        else:

            fp += 1

    fn = len(targets)-len(matched)

    return tp, fp, fn


def calculate_metrics(tp, fp, fn):

    precision = (
        tp/(tp+fp) if tp+fp else 0
    )

    recall = (
        tp/(tp+fn) if tp+fn else 0
    )

    f1 = (
        2*precision*recall/(precision+recall)
        if precision+recall else 0
    )

    return precision, recall, f1


print("Diagnostic evaluation functions ready.")

In [ ]:
EXPERIMENTS = [
    "standard",
    "high_resolution",
    "maximum_resolution",
    "high_sensitivity",
    "region_assisted"
]

experiment_records = []

# Warm up the models before measuring latency.
engine.set_profile("standard")

warmup = np.zeros(
    (640, 640, 3),
    dtype=np.uint8
)

_ = engine.process_image(
    warmup,
    mode="combined"
)

for profile in EXPERIMENTS:

    engine.set_profile(profile)

    # Helmet settings vary between profiles.
    # Seatbelt settings remain identical, so
    # evaluate seatbelt only under standard mode.

    subset = (
        diagnostic_df
        if profile == "standard"
        else diagnostic_df[
            diagnostic_df["task"] == "helmet"
        ]
    )

    for _, row in tqdm(
        subset.iterrows(),
        total=len(subset),
        desc=profile
    ):

        task = row["task"]

        image = cv2.imread(row["image_path"])

        assert image is not None

        ground = ground_truth_objects(
            image,
            row["label_path"]
        )

        result = engine.process_image(
            image,
            mode=task,
            use_context=True
        )

        predictions = result["detections"]

        for cls in [0, 1]:

            tp, fp, fn = match_detections(
                ground,
                predictions,
                cls
            )

            experiment_records.append({
                "Profile": profile,
                "Task": task,
                "Image": Path(
                    row["image_path"]
                ).name,
                "Class ID": cls,
                "TP": tp,
                "FP": fp,
                "FN": fn,
                "Latency ms":
                    result["summary"]["processing_ms"],
                "Crop detections":
                    result["summary"]["crop_detections"]
            })

    # Preserve progress after each experiment.

    pd.DataFrame(experiment_records).to_csv(
        TABLES_DIR /
        "diagnostic_experiment_records.csv",
        index=False
    )

print("\nALL DIAGNOSTIC EXPERIMENTS COMPLETED.")

In [ ]:
records_df = pd.DataFrame(
    experiment_records
)

assert not records_df.empty

class_results = []

for keys, group in records_df.groupby(
    ["Profile", "Task", "Class ID"]
):

    profile, task, cls = keys

    tp = int(group["TP"].sum())
    fp = int(group["FP"].sum())
    fn = int(group["FN"].sum())

    p, r, f1 = calculate_metrics(
        tp, fp, fn
    )

    class_results.append({
        "Profile": profile,
        "Task": task,
        "Class ID": cls,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": p,
        "Recall": r,
        "F1": f1
    })

diagnostic_class_df = pd.DataFrame(
    class_results
)

comparison = []

for keys, group in diagnostic_class_df.groupby(
    ["Profile", "Task"]
):

    profile, task = keys

    violation = group[
        group["Class ID"] == 1
    ]

    assert len(violation) == 1

    violation = violation.iloc[0]

    latency = records_df[
        (records_df["Profile"] == profile)
        & (records_df["Task"] == task)
    ]

    latency = latency.drop_duplicates(
        subset=["Image"]
    )

    comparison.append({
        "Profile": profile,
        "Task": task,
        "Images": len(latency),
        "Precision": group["Precision"].mean(),
        "Recall": group["Recall"].mean(),
        "Macro F1": group["F1"].mean(),
        "Violation Precision": violation["Precision"],
        "Violation Recall": violation["Recall"],
        "Violation F1": violation["F1"],
        "Mean Latency ms": latency["Latency ms"].mean()
    })

profile_df = pd.DataFrame(comparison)

profile_df.to_csv(
    TABLES_DIR / "profile_comparison.csv",
    index=False
)

diagnostic_class_df.to_csv(
    TABLES_DIR / "diagnostic_class_metrics.csv",
    index=False
)

metrics = [
    "Precision",
    "Recall",
    "Macro F1",
    "Violation Precision",
    "Violation Recall",
    "Violation F1"
]

shown = profile_df.copy()

shown[metrics] *= 100

display(shown.round(2))

helmet_results = shown[
    shown["Task"] == "helmet"
]

ax = helmet_results.set_index(
    "Profile"
)[
    [
        "Precision",
        "Recall",
        "Macro F1",
        "Violation F1"
    ]
].plot(
    kind="bar",
    figsize=(14, 6)
)

ax.set_title(
    "Helmet Inference Profile Comparison"
)

ax.set_ylabel("Performance (%)")
ax.set_ylim(0, 110)

for container in ax.containers:

    ax.bar_label(
        container,
        fmt="%.1f",
        fontsize=8
    )

plt.xticks(rotation=20)
plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "helmet_profile_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("\nPROFILE COMPARISON COMPLETED.")

In [ ]:
from google.colab import files

uploaded = files.upload()

EXTERNAL_IMAGES = []

for index, (filename, content) in enumerate(
    uploaded.items()
):

    suffix = Path(filename).suffix.lower()

    if suffix not in IMAGE_EXTENSIONS:

        print("Unsupported file:", filename)
        continue

    target = (
        UPLOAD_DIR /
        f"{index:03d}_{Path(filename).name}"
    )

    array = np.frombuffer(
        content,
        dtype=np.uint8
    )

    image = cv2.imdecode(
        array,
        cv2.IMREAD_COLOR
    )

    if image is None:

        print("Invalid image:", filename)
        continue

    if target.exists():

        assert target.read_bytes() == content

    else:

        target.write_bytes(content)

    EXTERNAL_IMAGES.append(target)

assert EXTERNAL_IMAGES, (
    "Upload at least one valid image."
)

print("External images:", len(EXTERNAL_IMAGES))

for path in EXTERNAL_IMAGES:
    print(path.name)

In [ ]:
def show_image(ax, image, title):

    ax.imshow(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )

    ax.set_title(title)
    ax.axis("off")


external_records = []

# Limit detailed comparison on CPU.
MAX_DETAILED_IMAGES = 6

for path in EXTERNAL_IMAGES[:MAX_DETAILED_IMAGES]:

    image = cv2.imread(str(path))

    context = engine.raw_predict(
        "context",
        image,
        confidence=0.25
    )

    low_conf = engine.raw_predict(
        "helmet",
        image,
        size=640,
        confidence=0.05
    )

    fig, axes = plt.subplots(
        2, 3,
        figsize=(18, 11)
    )

    show_image(
        axes[0, 0],
        image,
        "Original Image"
    )

    show_image(
        axes[0, 1],
        engine.draw_raw(image, context),
        "Vehicle and Person Context"
    )

    engine.set_profile("standard")

    standard = engine.process_image(
        image,
        mode="helmet"
    )

    show_image(
        axes[0, 2],
        standard["annotated_image"],
        "Standard: 640 / 0.30"
    )

    show_image(
        axes[1, 0],
        engine.draw_raw(image, low_conf),
        "Raw Diagnostic: 640 / 0.05"
    )

    engine.set_profile("high_sensitivity")

    sensitive = engine.process_image(
        image,
        mode="helmet"
    )

    show_image(
        axes[1, 1],
        sensitive["annotated_image"],
        "High Sensitivity: 800 / 0.20"
    )

    engine.set_profile("region_assisted")

    region = engine.process_image(
        image,
        mode="helmet"
    )

    show_image(
        axes[1, 2],
        region["annotated_image"],
        "Region-Assisted"
    )

    for result in [
        standard,
        sensitive,
        region
    ]:

        external_records.append({
            "Image": path.name,
            **result["summary"]
        })

    plt.suptitle(path.name)
    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR /
        f"diagnostic_{path.stem}.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print("\nRaw helmet predictions:")

    display(
        pd.DataFrame(low_conf)[
            ["label", "confidence", "bbox"]
        ].round(3)
        if low_conf else
        pd.DataFrame(
            columns=["label", "confidence", "bbox"]
        )
    )

    print(
        "Motorcycles detected:",
        sum(
            d["class_id"] == 3
            for d in context
        )
    )

    print(
        "Persons detected:",
        sum(
            d["class_id"] == 0
            for d in context
        )
    )

external_df = pd.DataFrame(
    external_records
)

external_df.to_csv(
    TABLES_DIR /
    "external_profile_diagnostics.csv",
    index=False
)

display(external_df)

engine.set_profile("standard")

In [ ]:
ACTIVE_PROFILE = "standard"

assert ACTIVE_PROFILE in PROFILES

engine.set_profile(ACTIVE_PROFILE)

external_summaries = []
external_detections = []

for path in tqdm(
    EXTERNAL_IMAGES,
    desc="External combined testing"
):

    image = cv2.imread(str(path))

    assert image is not None

    result = engine.process_image(
        image,
        mode="combined",
        use_context=True
    )

    external_summaries.append({
        "Image": path.name,
        **result["summary"]
    })

    for detection in result["detections"]:

        external_detections.append({
            "Image": path.name,
            **detection
        })

    output = (
        OUTPUT_DIR /
        f"external_{path.stem}.jpg"
    )

    assert cv2.imwrite(
        str(output),
        result["annotated_image"]
    )

    fig, axes = plt.subplots(
        1, 2,
        figsize=(14, 6)
    )

    show_image(
        axes[0],
        image,
        "Original"
    )

    show_image(
        axes[1],
        result["annotated_image"],
        "Combined Detection"
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)


external_summary_df = pd.DataFrame(
    external_summaries
)

external_summary_df.to_csv(
    TABLES_DIR /
    "external_combined_summary.csv",
    index=False
)

(
    REPORTS_DIR /
    "external_combined_detections.json"
).write_text(
    json.dumps(
        external_detections,
        indent=4
    )
)

display(external_summary_df)

print("\nEXTERNAL IMAGE TESTING COMPLETED.")

In [ ]:
uploaded_video = files.upload()

assert len(uploaded_video) == 1, (
    "Upload one traffic video."
)

video_name, video_content = next(
    iter(uploaded_video.items())
)

suffix = Path(video_name).suffix.lower()

assert suffix in {
    ".mp4", ".avi", ".mov", ".mkv", ".webm"
}

VIDEO_PATH = UPLOAD_DIR / (
    f"video_{datetime.now():%H%M%S%f}{suffix}"
)

VIDEO_PATH.write_bytes(video_content)

cap = cv2.VideoCapture(str(VIDEO_PATH))

assert cap.isOpened(), "Cannot open uploaded video."

SOURCE_FPS = float(
    cap.get(cv2.CAP_PROP_FPS)
)

TOTAL_FRAMES = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

VIDEO_WIDTH = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

VIDEO_HEIGHT = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

cap.release()

assert np.isfinite(SOURCE_FPS)
assert SOURCE_FPS > 0
assert TOTAL_FRAMES > 0

DURATION = TOTAL_FRAMES / SOURCE_FPS

video_info = {
    "Filename": video_name,
    "FPS": SOURCE_FPS,
    "Total Frames": TOTAL_FRAMES,
    "Width": VIDEO_WIDTH,
    "Height": VIDEO_HEIGHT,
    "Duration Seconds": DURATION
}

display(
    pd.DataFrame([video_info]).T
)

(
    REPORTS_DIR /
    "video_source_information.json"
).write_text(
    json.dumps(video_info, indent=4)
)

print("\nVIDEO INFORMATION VERIFIED.")

In [ ]:
NUMBER_OF_FRAMES = 12

frame_indices = sorted(set(

    np.linspace(
        0,
        TOTAL_FRAMES-1,
        num=min(
            NUMBER_OF_FRAMES,
            TOTAL_FRAMES
        )
    ).astype(int).tolist()

))

cap = cv2.VideoCapture(str(VIDEO_PATH))

frame_records = []

try:

    for frame_index in tqdm(
        frame_indices,
        desc="Extracting diagnostic frames"
    ):

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_index
        )

        ok, frame = cap.read()

        if not ok:

            print(
                "Unreadable frame:",
                frame_index
            )

            continue

        frame_path = (
            FRAMES_DIR /
            f"frame_{frame_index:06d}.png"
        )

        assert cv2.imwrite(
            str(frame_path),
            frame
        )

        frame_records.append({

            "frame": int(frame_index),

            "timestamp_seconds":
                frame_index/SOURCE_FPS,

            "image_path": str(
                frame_path.relative_to(PROJECT_ROOT)
            ),

            "mode": "combined"

        })

finally:

    cap.release()

frame_manifest_df = pd.DataFrame(
    frame_records
)

assert not frame_manifest_df.empty

FRAME_MANIFEST_CSV = (
    TABLES_DIR /
    "video_frame_manifest.csv"
)

frame_manifest_df.to_csv(
    FRAME_MANIFEST_CSV,
    index=False
)

display(frame_manifest_df)

print(
    "Extracted frames:",
    len(frame_manifest_df)
)

print("Frame manifest:", FRAME_MANIFEST_CSV)

In [ ]:
rows = int(
    np.ceil(len(frame_manifest_df)/3)
)

fig, axes = plt.subplots(
    rows, 3,
    figsize=(16, rows*4),
    squeeze=False
)

for ax in axes.flat:
    ax.axis("off")

for i, row in frame_manifest_df.iterrows():

    image = cv2.imread(
        str(PROJECT_ROOT / row["image_path"])
    )

    show_image(
        axes.flat[i],
        image,
        f'Frame {row["frame"]} | '
        f'{row["timestamp_seconds"]:.2f}s'
    )

plt.suptitle(
    "Representative Frames Across Entire Video"
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "original_video_frame_samples.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
VIDEO_PROFILES = [
    "standard",
    "high_sensitivity",
    "region_assisted"
]

video_frame_results = []

for profile in VIDEO_PROFILES:

    engine.set_profile(profile)

    for _, row in tqdm(
        frame_manifest_df.iterrows(),
        total=len(frame_manifest_df),
        desc=f"Video frames: {profile}"
    ):

        image_path = (
            PROJECT_ROOT /
            row["image_path"]
        )

        frame = cv2.imread(str(image_path))

        assert frame is not None

        result = engine.process_image(
            frame,
            mode="combined"
        )

        video_frame_results.append({

            "Frame": int(row["frame"]),

            "Timestamp":
                float(row["timestamp_seconds"]),

            **result["summary"]
        })

        output_path = (
            OUTPUT_DIR /
            f'{profile}_frame_{int(row["frame"]):06d}.jpg'
        )

        assert cv2.imwrite(
            str(output_path),
            result["annotated_image"]
        )

    pd.DataFrame(video_frame_results).to_csv(
        TABLES_DIR /
        "video_frame_profile_results.csv",
        index=False
    )


video_frame_df = pd.DataFrame(
    video_frame_results
)

video_comparison = video_frame_df.groupby(
    "profile"
).agg(

    Frames=("Frame", "count"),

    Total_Detections=(
        "total_detections", "sum"
    ),

    Helmet_Violation_Detections=(
        "possible_helmet_violations", "sum"
    ),

    Seatbelt_Violation_Detections=(
        "possible_seatbelt_violations", "sum"
    ),

    Unverified=(
        "unverified_detections", "sum"
    ),

    Crop_Detections=(
        "crop_detections", "sum"
    ),

    Mean_Latency_ms=(
        "processing_ms", "mean"
    )

).reset_index()

display(video_comparison.round(2))

video_comparison.to_csv(
    TABLES_DIR /
    "video_profile_comparison.csv",
    index=False
)

print("\nVIDEO PROFILE COMPARISON COMPLETED.")

In [ ]:
DISPLAY_FRAMES = min(
    6,
    len(frame_manifest_df)
)

positions = np.linspace(
    0,
    len(frame_manifest_df)-1,
    DISPLAY_FRAMES
).astype(int)

selected_frames = (
    frame_manifest_df.iloc[positions]
    .reset_index(drop=True)
)

fig, axes = plt.subplots(
    len(selected_frames),
    4,
    figsize=(19, len(selected_frames)*3.5),
    squeeze=False
)

for i, row in selected_frames.iterrows():

    frame_index = int(row["frame"])

    original = cv2.imread(
        str(PROJECT_ROOT / row["image_path"])
    )

    show_image(
        axes[i, 0],
        original,
        f'Original | {row["timestamp_seconds"]:.2f}s'
    )

    for j, profile in enumerate(
        VIDEO_PROFILES,
        start=1
    ):

        path = (
            OUTPUT_DIR /
            f"{profile}_frame_{frame_index:06d}.jpg"
        )

        image = cv2.imread(str(path))

        assert image is not None, path

        show_image(
            axes[i, j],
            image,
            profile.replace("_", " ").title()
        )

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "video_frame_visual_comparison.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from video_processor import process_video

# Keep standard until we have reviewed the
# diagnostic results from alternative profiles.

VIDEO_PROFILE = "standard"

engine.set_profile(VIDEO_PROFILE)

TARGET_INFERENCE_FPS = (
    5 if torch.cuda.is_available()
    else 2
)

FRAME_STRIDE = max(
    1,
    round(SOURCE_FPS / TARGET_INFERENCE_FPS)
)

VIDEO_OUTPUT = (
    OUTPUT_DIR /
    f"refined_{VIDEO_PROFILE}_"
    f"{datetime.now():%H%M%S%f}.mp4"
)

progress = tqdm(
    total=TOTAL_FRAMES,
    desc="Processing complete video"
)

def update_progress(current, total):

    progress.update(
        max(
            0,
            min(current, TOTAL_FRAMES)
            - progress.n
        )
    )


try:

    video_report = process_video(

        engine=engine,

        input_path=VIDEO_PATH,

        output_path=VIDEO_OUTPUT,

        mode="combined",

        frame_stride=FRAME_STRIDE,

        max_frames=None,

        progress_callback=update_progress

    )

    progress.update(
        max(0, TOTAL_FRAMES-progress.n)
    )

finally:

    progress.close()


video_report["profile"] = VIDEO_PROFILE

video_report["target_inference_fps"] = (
    TARGET_INFERENCE_FPS
)

video_report["source_duration_seconds"] = (
    DURATION
)

video_report["expected_processed_frames"] = (
    (TOTAL_FRAMES + FRAME_STRIDE-1)
    // FRAME_STRIDE
)

video_report["frame_count_matches"] = (
    video_report["processed_frames"]
    == video_report["expected_processed_frames"]
)

(
    REPORTS_DIR /
    "complete_video_processing_report.json"
).write_text(
    json.dumps(video_report, indent=4)
)

print(json.dumps(video_report, indent=4))

assert VIDEO_OUTPUT.is_file()

print("\nVIDEO PROCESSING FINISHED.")
print("Output:", VIDEO_OUTPUT)

In [ ]:
import subprocess

from IPython.display import Video

PLAYABLE_VIDEO = VIDEO_OUTPUT.with_name(
    VIDEO_OUTPUT.stem + "_h264.mp4"
)

if shutil.which("ffmpeg"):

    if not PLAYABLE_VIDEO.exists():

        subprocess.run(
            [
                "ffmpeg",
                "-n",
                "-i", str(VIDEO_OUTPUT),

                "-c:v", "libx264",
                "-preset", "fast",
                "-crf", "23",

                "-pix_fmt", "yuv420p",
                "-movflags", "+faststart",

                str(PLAYABLE_VIDEO)
            ],
            check=True,
            capture_output=True
        )

    display(
        Video(
            str(PLAYABLE_VIDEO),
            embed=True,
            width=850
        )
    )

else:

    print("FFmpeg is not available.")

    print(
        "Open the output video from Drive:",
        VIDEO_OUTPUT
    )

In [ ]:
VIDEO_CSV = VIDEO_OUTPUT.with_suffix(".csv")

assert VIDEO_CSV.is_file()

video_stats_df = pd.read_csv(VIDEO_CSV)

assert not video_stats_df.empty

columns = [
    "with_helmet",
    "without_helmet",
    "with_seatbelt",
    "without_seatbelt",
    "possible_helmet_violations",
    "possible_seatbelt_violations",
    "unverified_detections",
    "total_detections"
]

columns = [
    c for c in columns
    if c in video_stats_df.columns
]

display(video_stats_df.head())

summary = pd.DataFrame({

    "Detection Category": columns,

    "Detection Instances": [
        int(video_stats_df[c].sum())
        for c in columns
    ],

    "Frames Containing Detection": [
        int((video_stats_df[c] > 0).sum())
        for c in columns
    ]
})

display(summary)

summary.to_csv(
    TABLES_DIR /
    "complete_video_detection_summary.csv",
    index=False
)

fig, axes = plt.subplots(
    2, 1,
    figsize=(15, 10)
)

for column in [
    "with_helmet",
    "without_helmet",
    "with_seatbelt",
    "without_seatbelt"
]:

    if column in video_stats_df.columns:

        axes[0].plot(
            video_stats_df["timestamp_seconds"],
            video_stats_df[column],
            label=column.replace(
                "_", " "
            ).title()
        )

axes[0].set_title(
    "Safety Equipment Detections Over Time"
)

for column in [
    "possible_helmet_violations",
    "possible_seatbelt_violations",
    "unverified_detections"
]:

    if column in video_stats_df.columns:

        axes[1].plot(
            video_stats_df["timestamp_seconds"],
            video_stats_df[column],
            label=column.replace(
                "_", " "
            ).title()
        )

axes[1].set_title(
    "Context-Supported and Unverified Detections"
)

for ax in axes:

    ax.set_xlabel("Video Time (seconds)")

    ax.set_ylabel("Detections per Frame")

    ax.grid(alpha=0.2)
    ax.legend()

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "complete_video_detection_statistics.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Processed frames:", len(video_stats_df))

print(
    "Last analyzed timestamp:",
    video_stats_df["timestamp_seconds"].max()
)

print(
    "Frames with detections:",
    int(
        (
            video_stats_df["total_detections"] > 0
        ).sum()
    )
)

In [ ]:
verification_records = []

# Verify original model weights.

for task, cfg in APP_CONFIG["models"].items():

    path = PROJECT_ROOT / cfg["weights"]

    verified = (
        path.is_file()
        and sha256_file(path)
        == cfg["model_sha256"]
    )

    verification_records.append({
        "Component": f"{task} model",
        "Verified": verified
    })

# Verify original source files.

for path, expected_hash in (
    ORIGINAL_SOURCE_HASHES.items()
):

    verified = (
        Path(path).is_file()
        and sha256_file(Path(path))
        == expected_hash
    )

    verification_records.append({
        "Component": Path(path).name,
        "Verified": verified
    })

verification_records.append({

    "Component": "Original app configuration",

    "Verified": (
        sha256_file(APP_CONFIG_PATH)
        == ORIGINAL_CONFIG_HASH
    )
})

verification_records.append({

    "Component": "Refined engine",

    "Verified": REFINED_PATH.is_file()
})

verification_df = pd.DataFrame(
    verification_records
)

display(verification_df)

assert verification_df["Verified"].all()

verification_df.to_csv(
    TABLES_DIR /
    "phase9b_integrity_verification.csv",
    index=False
)

# Preserve available profiles separately.

profiles_metadata = {

    "default_profile": "standard",

    "profiles": PROFILES,

    "experimental_profiles_validated": False,

    "original_weights_unchanged": True,

    "requires_human_review": True
}

PROFILE_CONFIG_PATH = (
    REPORTS_DIR /
    "inference_profiles.json"
)

PROFILE_CONFIG_PATH.write_text(
    json.dumps(
        profiles_metadata,
        indent=4
    )
)

# Record completed operations.

report = {

    "project": "TrafficGuard AI",

    "phase": "9B",

    "session": SESSION_ID,

    "device": str(DEVICE),

    "training_performed": False,

    "original_models_preserved": True,

    "original_source_files_preserved": True,

    "original_configuration_preserved": True,

    "diagnostic_images": (
        len(diagnostic_df)
        if "diagnostic_df" in globals()
        else 0
    ),

    "profile_experiments_completed": (
        not profile_df.empty
        if "profile_df" in globals()
        else False
    ),

    "external_images_uploaded": len(
        globals().get("EXTERNAL_IMAGES", [])
    ),

    "csv_batch_processed": (
        int(
            (
                batch_result_df["Status"]
                == "PROCESSED"
            ).sum()
        )
        if "batch_result_df" in globals()
        and "Status" in batch_result_df.columns
        else 0
    ),

    "video_frames_extracted": (
        len(frame_manifest_df)
        if "frame_manifest_df" in globals()
        else 0
    ),

    "video_report": globals().get(
        "video_report", {}
    ),

    "default_profile": "standard",

    "final_profile_selection":
        "Pending diagnostic review",

    "limitations": [

        "Diagnostic sample metrics are not "
        "independent test metrics.",

        "Region-assisted inference is heuristic.",

        "Context association may miss "
        "occluded or distant riders.",

        "Video counts represent detection instances.",

        "Unique violation tracking is not implemented.",

        "CSV count matching does not establish "
        "bounding-box accuracy.",

        "Experimental inference settings require "
        "further verification."

    ]
}

if "profile_df" in globals():

    report["profile_comparison"] = json.loads(
        profile_df.to_json(orient="records")
    )

REPORT_PATH = (
    REPORTS_DIR /
    "phase9b_refinement_report.json"
)

REPORT_PATH.write_text(
    json.dumps(report, indent=4)
)

print(json.dumps(report, indent=4))

print("\n" + "="*65)

print("PHASE 9B FILE VERIFICATION PASSED")

print("="*65)

print("Original weights unchanged: YES")
print("Original source files unchanged: YES")
print("Original configuration unchanged: YES")

print("Refined engine:", REFINED_PATH)

print("Results directory:", SESSION_DIR)

print("Final report:", REPORT_PATH)

print("\nFINAL PROFILE ACCEPTANCE: PENDING REVIEW.")